<a href="https://colab.research.google.com/github/SamiNt67/Practica2_IA/blob/main/Copia_de_pr%C3%A1ctica_2_1_sentimientos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qué es el análisis de sentimiento**

Es una tarea de clasificación de texto. Dado un texto, el modelo decide si la intención utilizada es positiva, negativa o neutral.

No es "**entender**" el texto como lo haría una persona. Es detectar patrones de palabras y contexto que, estadísticamente, se asocian a cada categoría.

Para esta práctica en español, el modelo más adecuado es **RoBERTuito**. Es un modelo entrenado específicamente con **500 millones de tweets en español**, lo que le permite entender jerga, emojis, abreviaturas y texto informal.

Está diseñado para redes sociales, que es donde el análisis de sentimiento tiene más aplicación real. Y funciona bien en tareas de sentimiento, odio e ironía en español.

RoBERTuito no se entrenó desde cero. Es un modelo preentrenado: alguien invirtió muchísimo tiempo y recursos en entrenarlo con textos masivos para que "entienda" el español.

Luego ese modelo base se ajustó (mediante fine-tuning) con un conjunto más pequeño de tweets etiquetados como positivos, negativos o neutros. El resultado es un modelo especializado en análisis de sentimiento en español, listo para usar.

Esto es importante porque explica por qué tú no entrenas nada. Usas un modelo que ya existe, ya está entrenado, y solo lo llamas desde Colab. Es exactamente lo que haría un desarrollador profesional: **no reinventar la rueda, usar lo que ya funciona**.

# **Hugging Face**

Hugging Face es la plataforma donde viven estos modelos. Su función **pipeline** es una abstracción: tú le dices "quiero realizar un análisis de sentimiento" y "usa este modelo", y ella se encarga de todo lo demás (preprocesar el texto, pasarlo por el modelo, devolver el resultado).

Solo se necesita entender que pipeline es el "botón" que conecta el texto con un modelo ya entrenado.

El modelo devuelve dos cosas:

    Una etiqueta: POS, NEG o NEU

    Una confianza (score), un número entre 0 y 1

La confianza no es lo mismo que **"acierto"**. Es la seguridad del modelo en su propia predicción. Un score de 0.99 no garantiza que tenga razón; un score de 0.51 en una frase ambigua es normal y no es un fallo.

El modelo no lee intenciones, lee patrones. El guiño o el signo de interrogación alteran el contexto que el modelo aprendió durante el preentrenamiento, y eso mueve la predicción.

In [ ]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")
# carga un modelo de inteligencia artificial
# entrenado para analizar el sentimiento
# de textos en español, con especial enfoque
# en lenguaje informal y redes sociales

# === este es el texto  que se clasifica ===
texto = "Me encanta este módulo de IA"
# =====================================

resultado = clasificador(texto)
print(resultado)

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
resultado = clasificador("Esto es horrible.")
print(resultado)

[{'label': 'NEG', 'score': 0.9667719602584839}]


In [ ]:
# clasifica un texto que sea negativo

texto = "Este producto es horrible."
resultado = clasificador(texto)
print(resultado)

In [ ]:
# clasifica un texto neutro
texto = "Hoy hace un día normal."
resultado = clasificador(texto)
print(resultado)

In [ ]:
# clasifica un texto irónico
texto = "Qué maravilla, se me ha vuelto a romper el ordenador."
resultado = clasificador(texto)
print(resultado)

In [ ]:
# clasifica un texto usando emojis

texto = "Me encanta este producto 😍❤️"
resultado = clasificador(texto)
print(resultado)

**¿Cómo funciona el análisis de sentimientos? ¿Qué límites tiene?**

---



¿Cómo funciona el análisis de sentimientos y qué límites tiene?
El análisis de sentimientos que hemos visto es básicamente una clasificación de texto para así poder estimar que sentimiento quiere transmitir. Funciona analizando un mensaje y, a partir de eso, clasificarlo como positivo, negativo o neutral según las palabras y el contexto que contiene.

En esta práctica se nos pide que utilicemos RoBERTuito (por lo que he visto es un modelo especializado en español y entrenado con aproximadamente 500 millones de tweets). Por eso, es capaz de trabajar especialmente bien con jerga, emojis, abreviaturas y lenguaje informal. RobERTuito ya está preentrenado con tweets etiquetados como positivos, negativos y neutros.

¿Qué límites tiene?
- Su primer y más evidente límite, claramente es que no entiende realmente como lo hacemos los humanos, tan solo emula un análisis y entendimiento. Por eso también, realmente no comprende las emociones que está clasificand.
- No es humano, le falta razonamiento por lo que puede interpretar mal frases ambiguas o con doble sentido o tener dificultades para detectar la ironía y el sarcasmo.
- Cuando hay alguna expresión no habitual, no lo entenderá si no ha sido entrenada con esta.

# **Práctica**

Implementa en python un analizador de comentarios. Los comentarios positivos y neutros serán aceptados, salvo que contengan alguna palabra de la lista negra de palabras. Los comentarios negativos que contengan alguna palabra de la lista negra o que tengan un score de más de 0.5 serán rechazadas.

In [ ]:
# pon aquí el código de tu analizador
import csv
import json
import os
from typing import Dict, Any

from transformers import pipeline

MODEL_NAME = "pysentimiento/robertuito-sentiment-analysis"
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
BLACKLIST_PATH = os.path.join(BASE_DIR, "lista_negra.txt")
DATASET_PATH = os.path.join(BASE_DIR, "dataset_comentarios.csv")


def cargar_lista_negra(ruta: str = BLACKLIST_PATH):
    with open(ruta, "r", encoding="utf-8") as archivo:
        palabras = {linea.strip().lower() for linea in archivo if linea.strip()}
    return palabras


BLACKLIST = cargar_lista_negra()


def normalizar_texto(texto: str) -> str:
    return " ".join(texto.lower().split())


def contiene_palabra_negra(texto: str) -> bool:
    texto_normalizado = normalizar_texto(texto)
    palabras = set(texto_normalizado.split())
    return bool(palabras & BLACKLIST)


def cargar_dataset(ruta: str = DATASET_PATH):
    with open(ruta, "r", encoding="utf-8", newline="") as archivo:
        lector = csv.DictReader(archivo)
        return list(lector)



def evaluar_comentario(comentario: str, clasificador) -> Dict[str, Any]:
    texto = (comentario or "").strip()
    if not texto:
        return {
            "comentario": texto,
            "label": "NEU",
            "score": 0.0,
            "aceptado": False,
            "palabra_negra": False,
            "motivo": "El comentario está vacío. No se puede evaluar.",
        }

    prediccion = clasificador(texto, truncation=True)[0]
    label = prediccion["label"]
    score = float(prediccion["score"])
    palabra_negra = contiene_palabra_negra(texto)

    if label in {"POS", "NEU"}:
        aceptado = not palabra_negra
        motivo = (
            "Comentario positivo o neutro aceptado porque no contiene palabras de la lista negra."
            if aceptado
            else "Comentario positivo o neutro rechazado porque contiene una palabra de la lista negra."
        )

    elif label == "NEG":
        if palabra_negra:
            aceptado = False
            motivo = "Comentario negativo rechazado porque contiene una palabra de la lista negra."
        elif score > 0.5:
            aceptado = False
            motivo = f"Comentario negativo rechazado por score {score:.2f} > 0.5."
        else:
            aceptado = True
            motivo = "Comentario negativo aceptado: no contiene palabras de la lista negra y el score no supera 0.5."

    else:
        aceptado = not palabra_negra
        motivo = (
            "Comentario aceptado porque no contiene palabras de la lista negra."
            if aceptado
            else "Comentario rechazado porque contiene una palabra de la lista negra."
        )

    return {
        "comentario": comentario,
        "label": label,
        "score": round(score, 4),
        "aceptado": aceptado,
        "palabra_negra": palabra_negra,
        "motivo": motivo,
    }


def main():
    print("Analizador de comentarios con RoBERTuito")
    print("Escribe 'salir' para terminar.\n")

    clasificador = pipeline(
        "sentiment-analysis",
        model=MODEL_NAME,
    )

    print("Dataset de ejemplo cargado:")
    dataset = cargar_dataset()
    for fila in dataset:
        comentario = fila["comentario"]
        resultado = evaluar_comentario(comentario, clasificador)
        print(json.dumps(resultado, ensure_ascii=False, indent=2))
        print("-" * 60)

    print("\nIntroduce tus propios comentarios:")
    while True:
        comentario = input("Introduce un comentario: ").strip()
        if comentario.lower() in {"salir", "exit", "quit", "q"}:
            print("Fin del analizador.")
            break

        resultado = evaluar_comentario(comentario, clasificador)
        print(json.dumps(resultado, ensure_ascii=False, indent=2))
        print("-" * 60)


if __name__ == "__main__":
    main()
